In [1]:
!pip install "datasets<4.0"

  Attempting uninstall: fsspec
    Found existing installation: fsspec 2025.12.0
    Uninstalling fsspec-2025.12.0:
      Successfully uninstalled fsspec-2025.12.0
  Attempting uninstall: dill━━━━━━━━━━━━━━━━━━━━ 0/4 [fsspec]
    Found existing installation: dill 0.4.1━ 0/4 [fsspec]
    Uninstalling dill-0.4.1:━━━━━━━━━━━━━━━━ 0/4 [fsspec]
      Successfully uninstalled dill-0.4.1━━━━━━━━━━━━━━━━━━━━━━━━━ 1/4 [dill]
  Attempting uninstall: multiprocessm━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1/4 [dill]
    Found existing installation: multiprocess 0.70.19━━━━━━━━━ 1/4 [dill]
    Uninstalling multiprocess-0.70.19:━━━━━━━━━━━━━━━━━━━━━━━━ 1/4 [dill]
      Successfully uninstalled multiprocess-0.70.19━━━━━━━━━━━ 1/4 [dill]
  Attempting uninstall: datasets╺━━━━━━━━━━━━━━━━━━━ 2/4 [multiprocess]
    Found existing installation: datasets 4.8.5━━━━━━━━━━━━━━━ 2/4 [multiprocess]
    Uninstalling datasets-4.8.5:━━━━━━╺━━━━━━━━━ 3/4 [datasets]
      Successfully uninstalled datasets-4.8.5━━━━━━━━━ 3/4 [da

In [2]:
import torch
import numpy as np
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    TrainingArguments,
    Trainer,
    DataCollatorForTokenClassification
)

In [3]:
!pip install evaluate
import evaluate

In [4]:
device = "cuda" if torch.cuda.is_available() else "cpu"
device

'cuda'

In [5]:
dataset = load_dataset("leondz/wnut_17", split="train[:1000]")
val_dataset = load_dataset("leondz/wnut_17", split="validation[:200]")

README.md:   0%|          | 0.00/9.05k [00:00<?, ?B/s]

wnut_17.py:   0%|          | 0.00/7.46k [00:00<?, ?B/s]

The repository for leondz/wnut_17 contains custom code which must be executed to correctly load the dataset. You can inspect the repository content at https://hf.co/datasets/leondz/wnut_17.
You can avoid this prompt in future by passing the argument `trust_remote_code=True`.

Do you wish to run the custom code? [y/N]  y


Generating train split:   0%|          | 0/3394 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1009 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1287 [00:00<?, ? examples/s]

In [7]:
label_list = dataset.features["ner_tags"].feature.names
print(f"{label_list}")

['O', 'B-corporation', 'I-corporation', 'B-creative-work', 'I-creative-work', 'B-group', 'I-group', 'B-location', 'I-location', 'B-person', 'I-person', 'B-product', 'I-product']


In [8]:
model_ckpt = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_ckpt)

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

In [12]:
def tokenize_and_align_labels(examples):
    tokenized_inputs = tokenizer(
        examples["tokens"],
        truncation=True,
        is_split_into_words=True,
        max_length=128
    )
    labels = []
    
    for i, label in enumerate(examples["ner_tags"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids = []
        for word_idx in word_ids:
            if word_idx is None:
                label_ids.append(-100)
            elif word_idx != previous_word_idx:
                label_ids.append(label[word_idx])
            else:
                label_ids.append(-100)
            previous_word_idx = word_idx
        labels.append(label_ids)
    tokenized_inputs["labels"] = labels
    return tokenized_inputs

encoded_train = dataset.map(tokenize_and_align_labels, batched=True)
encoded_val = val_dataset.map(tokenize_and_align_labels, batched= True)

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

In [ ]:
["Hugging", "Face"]
[1, 2]
B-ORG I-ORG
["hugg", "##ging"]

[CLS]
hugg
##ging
face
[SEP]


word_ids

[None, 0, 0, 1 , None]

In [13]:
model = AutoModelForTokenClassification.from_pretrained(
    model_ckpt,
    num_labels=len(label_list)
)

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForTokenClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [15]:
!pip install seqeval
metric = evaluate.load("seqeval")

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for seqeval: filename=seqeval-1.2.2-py3-none-any.whl size=16250 sha256=555cd50be9aabf0d782821fe370f192b0cda116e3ab8a042f4d51644b9776625
  Stored in directory: /root/.cache/pip/wheels/14/cf/a7/8f28ef376d707ff10e3922899482a2f23ef3002f8a952f47ac
Successfully built seqeval


In [16]:
def compute_metrics(eval_preds):
    
    logits, labels = eval_preds
    predictions = np.argmax(logits, axis=-1)
    true_predictions = [
        [label_list[p] for (p, l) in zip(prediction, label) if l != -100] for prediction, label in zip(predictions, labels)
    ]
    true_labels = [
        [label_list[l] for (p, l) in zip(prediction, label) if l != -100] for prediction, label in zip(predictions, labels)
    ]
    results = metric.compute(predictions=true_predictions, references=true_labels)
    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
        "accuracy": results["overall_accuracy"],
    }

In [17]:
data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=2,
    weight_decay=0.01,
    logging_steps=10,
    load_best_model_at_end=True,
    report_to="none"
    
)

In [19]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=encoded_train,
    eval_dataset=encoded_val,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

In [20]:
trainer.train()

/usr/local/lib/python3.13/dist-packages/torch/autograd/function.py:596: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,0.545173,1.096288,0.000000,0.000000,0.000000,0.902759
2,0.449559,1.000631,0.000000,0.000000,0.000000,0.902759


/usr/local/lib/python3.13/dist-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/usr/local/lib/python3.13/dist-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/torch/autograd/function.py:596: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]
/usr/local/lib/python3.13/dist-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/usr/local/lib/python3.13/dist-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=64, training_loss=1.1063041239976883, metrics={'train_runtime': 13.2375, 'train_samples_per_second': 151.086, 'train_steps_per_second': 4.835, 'total_flos': 27683547641760.0, 'train_loss': 1.1063041239976883, 'epoch': 2.0})

In [21]:
save_directory = "./saved_wnut_ner_model"
trainer.save_model(save_directory)
tokenizer.save_pretrained(save_directory)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./saved_wnut_ner_model/tokenizer_config.json',
 './saved_wnut_ner_model/tokenizer.json')

In [22]:
from transformers import pipeline

ner_pipeline = pipeline("ner", model=save_directory, tokenizer=save_directory, aggregation_strategy="simple")

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

In [23]:
text = "Twitter is testing a new feature in New York for its users"

print(f"{ner_pipeline(text)}")

[{'entity_group': 'LABEL_0', 'score': np.float32(0.8671815), 'word': 'twitter is testing a new feature in new york for its users', 'start': 0, 'end': 58}]
